## Data Cleaning Notebook

### Overview
This notebook performs the data cleaning phase of the Marketing Mix Modelling (MMM) project. The objective of this step is to ensure that the raw dataset is accurate, complete, and suitable for further analysis and modelling activities.

Marketing Mix Modelling relies heavily on the quality of data because inaccurate, inconsistent, or missing values can negatively impact the estimation of channel effectiveness, return on investment (ROI), and budget optimization decisions. Therefore, this notebook focuses on preparing a reliable dataset for subsequent feature engineering and modelling stages.

### Objectives
The main objectives of this notebook are to:

- Load the raw marketing dataset from the `data/raw` directory.
- Examine the dataset structure, dimensions, and data types.
- Identify and handle missing values.
- Detect and remove duplicate records.
- Correct data type inconsistencies.
- Convert date-related fields into appropriate formats.
- Detect and handle outliers in numerical variables.
- Validate advertising expenditure variables.
- Perform general data quality checks.
- Save the cleaned dataset to the `data/processed` directory.

### Dataset Description
The dataset contains information related to:

- Sales performance
- Product demand
- Advertising expenditures across multiple marketing channels
- Customer engagement metrics
- Market and economic indicators
- Promotional activities
- Seasonal and competitive factors

These variables will later be used to quantify marketing effectiveness and build an optimization model for advertising budget allocation.

### Expected Output
The cleaned dataset generated from this notebook will be saved as:

`data/processed/cleaned_marketing_mix_data.csv`

The resulting dataset will serve as the input for the Feature Engineering notebook.


In [4]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/raw/mmm_dataset.xlsx")

print(df.shape)
df.head()


(36, 25)


,month,country,brand_category,region_scope,tv_spend_lkr,facebook_instagram_spend_lkr,google_search_spend_lkr,youtube_spend_lkr,radio_spend_lkr,outdoor_spend_lkr,...,conversions,average_price_index,promotion_discount_pct,competitor_pressure_index_0_100,economic_sentiment_index_0_100,lkr_usd_exchange_rate,inflation_proxy_index,seasonal_demand_index,public_holiday_count,data_type
0,2023-01-01,Sri Lanka,FMCG - Packaged Foods,National,4274762,3890003,2467793,1836858,1700676,1813444,...,1210051,0.998,7.27,49.6,56.8,324.42,1.000,1.08,2,SYNTHETIC
1,2023-02-01,Sri Lanka,FMCG - Packaged Foods,National,4023574,5062635,2962576,2254198,1922384,2941194,...,664746,1.017,9.47,51.0,58.4,335.92,1.012,0.96,6,SYNTHETIC
2,2023-03-01,Sri Lanka,FMCG - Packaged Foods,National,5359091,3746337,4080535,2363737,1720670,2063917,...,380625,1.011,6.83,45.9,60.6,328.66,1.024,1.00,2,SYNTHETIC
3,2023-04-01,Sri Lanka,FMCG - Packaged Foods,National,5102550,3690115,3390558,2595121,1217450,3091430,...,1228413,1.017,21.52,31.2,54.5,325.60,1.037,1.19,6,SYNTHETIC
4,2023-05-01,Sri Lanka,FMCG - Packaged Foods,National,4972723,5333813,3240078,2343364,1568285,3525300,...,562986,1.014,1.95,53.0,56.5,330.03,1.049,1.10,3,SYNTHETIC


In [5]:
df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   month                            36 non-null     str    
 1   country                          36 non-null     str    
 2   brand_category                   36 non-null     str    
 3   region_scope                     36 non-null     str    
 4   tv_spend_lkr                     36 non-null     int64  
 5   facebook_instagram_spend_lkr     36 non-null     int64  
 6   google_search_spend_lkr          36 non-null     int64  
 7   youtube_spend_lkr                36 non-null     int64  
 8   radio_spend_lkr                  36 non-null     int64  
 9   outdoor_spend_lkr                36 non-null     int64  
 10  total_ad_spend_lkr               36 non-null     int64  
 11  sales_revenue_lkr                36 non-null     int64  
 12  units_sold                       36

,tv_spend_lkr,facebook_instagram_spend_lkr,google_search_spend_lkr,youtube_spend_lkr,radio_spend_lkr,outdoor_spend_lkr,total_ad_spend_lkr,sales_revenue_lkr,units_sold,impressions,clicks,conversions,average_price_index,promotion_discount_pct,competitor_pressure_index_0_100,economic_sentiment_index_0_100,lkr_usd_exchange_rate,inflation_proxy_index,seasonal_demand_index,public_holiday_count
count,3.600000e+01,3.600000e+01,3.600000e+01,3.600000e+01,3.600000e+01,3.600000e+01,3.600000e+01,3.600000e+01,36.000000,3.600000e+01,3.600000e+01,3.600000e+01,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,4.360118e+06,4.398434e+06,3.315365e+06,2.219826e+06,1.669667e+06,2.612717e+06,1.857613e+07,8.509194e+07,152975.500000,1.405974e+09,3.134397e+07,1.156680e+06,1.104639,11.637778,47.308333,48.733333,335.673056,1.243111,1.046667,4.277778
std,8.641465e+05,7.736826e+05,7.196457e+05,4.456069e+05,2.728982e+05,6.325274e+05,2.803202e+06,7.642101e+06,24743.534398,5.046953e+08,1.390015e+07,7.509588e+05,0.063258,6.005891,10.217391,12.315101,6.163758,0.156092,0.088544,1.845630
min,2.638465e+06,3.067341e+06,2.162014e+06,1.476724e+06,1.217450e+06,1.727984e+06,1.353442e+07,6.872716e+07,112715.000000,7.998427e+08,9.184483e+06,3.583500e+05,0.998000,1.950000,28.000000,26.100000,324.420000,1.000000,0.950000,1.000000
25%,3.846029e+06,3.732282e+06,2.907149e+06,1.882028e+06,1.459317e+06,2.052059e+06,1.597031e+07,8.134299e+07,137041.250000,9.912542e+08,2.256742e+07,6.157125e+05,1.052750,7.235000,38.850000,40.300000,329.940000,1.110750,0.987500,3.000000
50%,4.352655e+06,4.490288e+06,3.249230e+06,2.160448e+06,1.653602e+06,2.516048e+06,1.909133e+07,8.475952e+07,152151.000000,1.278593e+09,2.890221e+07,9.309500e+05,1.108000,11.115000,48.450000,46.400000,336.505000,1.234500,1.010000,4.000000
75%,4.920907e+06,4.952398e+06,3.907576e+06,2.458515e+06,1.898464e+06,3.176403e+06,2.108804e+07,8.882893e+07,169776.250000,1.841054e+09,4.180911e+07,1.332412e+06,1.156250,16.352500,54.525000,57.525000,339.322500,1.370000,1.085000,6.000000
max,6.020432e+06,6.030031e+06,5.096864e+06,3.178264e+06,2.169012e+06,3.690845e+06,2.273382e+07,1.007122e+08,203031.000000,2.651782e+09,6.380120e+07,3.101443e+06,1.214000,28.220000,65.600000,70.100000,348.070000,1.518000,1.240000,8.000000


In [6]:
missing_values = df.isnull().sum()

print(missing_values)


month                              0
country                            0
brand_category                     0
region_scope                       0
tv_spend_lkr                       0
facebook_instagram_spend_lkr       0
google_search_spend_lkr            0
youtube_spend_lkr                  0
radio_spend_lkr                    0
outdoor_spend_lkr                  0
total_ad_spend_lkr                 0
sales_revenue_lkr                  0
units_sold                         0
impressions                        0
clicks                             0
conversions                        0
average_price_index                0
promotion_discount_pct             0
competitor_pressure_index_0_100    0
economic_sentiment_index_0_100     0
lkr_usd_exchange_rate              0
inflation_proxy_index              0
seasonal_demand_index              0
public_holiday_count               0
data_type                          0
dtype: int64


There are no missing values

In [8]:
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates()


Duplicates: 0


In [ ]:
# convert date column
df['month'] = pd.to_datetime(df['month'])


In [10]:
df['month'] = pd.to_datetime(df['month'])

numeric_cols = [
    'tv_spend_lkr',
    'facebook_instagram_spend_lkr',
    'google_search_spend_lkr',
    'youtube_spend_lkr',
    'radio_spend_lkr',
    'outdoor_spend_lkr',
    'sales_revenue_lkr'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [11]:
numerical_cols = df.select_dtypes(include=np.number).columns

for col in numerical_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])


In [12]:
df['calculated_total_spend'] = (
    df['tv_spend_lkr']
    + df['facebook_instagram_spend_lkr']
    + df['google_search_spend_lkr']
    + df['youtube_spend_lkr']
    + df['radio_spend_lkr']
    + df['outdoor_spend_lkr']
)


In [15]:
df['spend_difference'] = (
    df['total_ad_spend_lkr']
    - df['calculated_total_spend']
)
print(df['spend_difference'] )


0     0.0
1     0.0
2     0.0
3    -1.0
4     1.0
5     1.0
6     0.0
7     0.0
8    -1.0
9     0.0
10    0.0
11   -1.0
12    1.0
13    1.0
14    0.0
15    0.0
16    0.0
17    1.0
18   -1.0
19    0.0
20    2.0
21    0.0
22    0.0
23    0.0
24   -1.0
25   -1.0
26    0.0
27   -1.0
28   -1.0
29   -1.0
30    0.0
31    0.0
32    1.0
33    0.0
34    0.0
35    0.0
Name: spend_difference, dtype: float64


In [ ]:
df['total_ad_spend_lkr'] = df['calculated_total_spend']

In [16]:
df.drop(
    ['calculated_total_spend',
     'spend_difference'],
    axis=1,
    inplace=True
)


In [17]:
output_path = "../data/processed/cleaned_marketing_mix_data.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved")


Cleaned dataset saved
